In [9]:
import pandas as pd
df = pd.read_csv(r"D:\Intership project\NetflixDataCleaningandAnalysisProject(SQL+Python)\data\netflix_titles.csv")
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [11]:
import pyodbc
from sqlalchemy import create_engine
import urllib

params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=DESKTOP-CJ2TM4F;"
    "DATABASE=NetflixDB;"
    "UID=sa;"
    "PWD=1234;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

df.to_sql('netflix', engine, if_exists='replace', index=False)
print("Data has been load in sql server")

result = pd.read_sql("SELECT TOP 5 * FROM netflix", engine)
print(result)

Data has been load in sql server
  show_id     type                  title         director  \
0      s1    Movie   Dick Johnson Is Dead  Kirsten Johnson   
1      s2  TV Show          Blood & Water             None   
2      s3  TV Show              Ganglands  Julien Leclercq   
3      s4  TV Show  Jailbirds New Orleans             None   
4      s5  TV Show           Kota Factory             None   

                                                cast        country  \
0                                               None  United States   
1  Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...   South Africa   
2  Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...           None   
3                                               None           None   
4  Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...          India   

           date_added  release_year rating   duration  \
0  September 25, 2021          2020  PG-13     90 min   
1  September 24, 2021          2021  TV-MA  2 Seasons

In [17]:
df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Unknown')
df['country'] = df['country'].fillna('Unknown')
df = df.dropna(subset=['date_added', 'rating', 'duration'])

df = df.drop_duplicates()

if df['date_added'].dtype == 'object':
    df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), format='%B %d, %Y')

if 'duration_number' not in df.columns:
    df['duration_number'] = df['duration'].str.extract(r'(\d+)', expand=False).astype(int)
    df['duration_type'] = df['duration'].str.extract(r'([a-zA-Z ]+)', expand=False).str.strip()
    df['duration_type'] = df['duration_type'].replace('Seasons', 'Season')

print("Shape:", df.shape)
print(df.isnull().sum())
print(df[['director', 'date_added', 'duration_number', 'duration_type']].head())

Shape: (8790, 14)
show_id            0
type               0
title              0
director           0
cast               0
country            0
date_added         0
release_year       0
rating             0
duration           0
listed_in          0
description        0
duration_number    0
duration_type      0
dtype: int64
          director date_added  duration_number duration_type
0  Kirsten Johnson 2021-09-25               90           min
1          Unknown 2021-09-24                2        Season
2  Julien Leclercq 2021-09-24                1        Season
3          Unknown 2021-09-24                1        Season
4          Unknown 2021-09-24                2        Season


In [19]:
df.to_sql('netflix', engine, if_exists='replace', index=False)
print("Cleaned data SQL Server mein successfully reload ho gaya!")

# Confirm karne ke liye check karo
result = pd.read_sql("SELECT TOP 5 * FROM netflix", engine)
print(result)

Cleaned data SQL Server mein successfully reload ho gaya!
  show_id     type                  title         director  \
0      s1    Movie   Dick Johnson Is Dead  Kirsten Johnson   
1      s2  TV Show          Blood & Water          Unknown   
2      s3  TV Show              Ganglands  Julien Leclercq   
3      s4  TV Show  Jailbirds New Orleans          Unknown   
4      s5  TV Show           Kota Factory          Unknown   

                                                cast        country  \
0                                            Unknown  United States   
1  Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...   South Africa   
2  Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...        Unknown   
3                                            Unknown        Unknown   
4  Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...          India   

  date_added  release_year rating   duration  \
0 2021-09-25          2020  PG-13     90 min   
1 2021-09-24          2021  TV-MA  2 Seasons  

In [21]:
query1 = """
SELECT type, COUNT(*) AS total_count
FROM netflix
GROUP BY type;
"""
result1 = pd.read_sql(query1, engine)
print(result1)

      type  total_count
0  TV Show         2664
1    Movie         6126


In [22]:
query2 = """
SELECT YEAR(date_added) AS year_added, COUNT(*) AS total_titles
FROM netflix
GROUP BY YEAR(date_added)
ORDER BY year_added;
"""
result2 = pd.read_sql(query2, engine)
print(result2)

    year_added  total_titles
0         2008             2
1         2009             2
2         2010             1
3         2011            13
4         2012             3
5         2013            11
6         2014            24
7         2015            82
8         2016           426
9         2017          1185
10        2018          1648
11        2019          2016
12        2020          1879
13        2021          1498


In [29]:
query3 = """
SELECT TRIM(value) AS country, COUNT(*) AS total_titles
FROM netflix
CROSS APPLY STRING_SPLIT(country, ',')
GROUP BY TRIM(value)
ORDER BY total_titles DESC
OFFSET 0 ROWS FETCH NEXT 10 ROWS ONLY;
"""
result3 = pd.read_sql(query3, engine)
print(result3)

          country  total_titles
0   United States          3681
1           India          1046
2         Unknown           829
3  United Kingdom           805
4          Canada           445
5          France           393
6           Japan           316
7           Spain           232
8     South Korea           231
9         Germany           226


In [26]:
query4 = """
SELECT TRIM(value) AS genre, COUNT(*) AS total_titles
FROM netflix
CROSS APPLY STRING_SPLIT(listed_in, ',')
GROUP BY TRIM(value)
ORDER BY total_titles DESC
OFFSET 0 ROWS FETCH NEXT 10 ROWS ONLY;
"""
result4 = pd.read_sql(query4, engine)
print(result4)

                      genre  total_titles
0      International Movies          2752
1                    Dramas          2426
2                  Comedies          1674
3    International TV Shows          1349
4             Documentaries           869
5        Action & Adventure           859
6                 TV Dramas           762
7        Independent Movies           756
8  Children & Family Movies           641
9           Romantic Movies           616


In [27]:
query5 = """
SELECT rating, COUNT(*) AS total_titles
FROM netflix
GROUP BY rating
ORDER BY total_titles DESC;
"""
result5 = pd.read_sql(query5, engine)
print(result5)

      rating  total_titles
0      TV-MA          3205
1      TV-14          2157
2      TV-PG           861
3          R           799
4      PG-13           490
5      TV-Y7           333
6       TV-Y           306
7         PG           287
8       TV-G           220
9         NR            79
10         G            41
11  TV-Y7-FV             6
12        UR             3
13     NC-17             3
